## Term count annotation

## 1. Libraries import

In [11]:
import pandas as pd
import numpy as np
import math
import re
from collections import Counter
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk import bigrams
from nltk import trigrams
from gensim.models import FastText
import ast

## 2. Data import

In [2]:
df = pd.read_csv('voies_par_longueur_annotation_cotationfr.csv')

## 3. Extracting natural features

In [3]:
# Build the dictionary of terms to extract from descriptions 

dico_terme_description = {
    'dévers': ['dévers', 'déversante', 'déversant', 'déversantes', 'déversants', 'déversé', 'déversée', 'déversées', 'déversés'],
    'surplomb': ['surplombant', 'surplombante', 'surplombé', 'surplomb', 'surplombs',
                  'bombé', 'bombés', 'bombée', 'bombées', 'toit', 'toits'],
    'dalle': ['dalle', 'dalles', 'adhérence', 'adhérences'],
    'fissure': ['fissure', 'fissures', 'faille', 'failles'],
    'épaule': ['épaule', 'épaules', 'épaulement', 'épaulements'],
    'dièdre': ['dièdre', 'dièdres'],
    'cheminée': ['cheminée', 'cheminées'],
    'vire': ['vire', 'vires', 'virette', 'virettes'],
    'terrasse': ['terrasse', 'terrasses', 'plateforme', 'plateformes'],
    'traversée': ['traversée', 'traversées', 'traverser', 'traverse', 'traversant'],
    'réglette': ['réglette', 'réglettes'],
    'bac': ['bac', 'grosse prise', 'grosses prises', 'bonne prise', 'bonnes prises'],
    'plat': ['plat', 'plats',
             'replat', 'replats'],
    'pince': ['pince', 'pinces'],
    'mono_doigt': ['mono-doigt', 'monodoigt', 'mono-doigts', 'monodoigts'],
    'bi_doigt': ['bi-doigt', 'bidoigt', 'bi-doigts', 'bidoigts'],
    'inversee': ['inversée', 'inversées', 'inversé', 'inversés', 'inverse', 'inverses'],
    'crochet': ['crochet', 'crochets'],
    'trou': ['trou', 'trous'],
    'goutte_eau': ["goutte d'eau", "gouttes d'eau"],
    'petite_prise': ['petite prise', 'petites prises'],
    'écailles': ['écaille', 'écailles'],
    'arête': ['arête', 'arêtes'],
    'pilier': ['pilier', 'piliers'],
    'couloir': ['couloir', 'couloirs'],
    'cannelure': ['cannelure', 'cannelures'],
    'éperon': ['éperon', 'éperons'],
    'contrefort': ['contrefort', 'contreforts'],
    'corniche': ['corniche', 'corniches'],
    'fissure-dièdre': ['fissure-dièdre', 'fissures-dièdres', 'fissure dièdre', 'fissures dièdres', 'fissures dièdre', 'fissures dièdres'],
}

In [4]:
# Function that compiles a regex for each category to extract

def compiler_dictionnaire(dico):
    """
    Compile one regex per dictionary category.
    - re.escape(t) : treats each term as literal text (important
      for terms with apostrophes or hyphens, e.g. "goutte d'eau", "mono-doigt",
      which contain characters with a special meaning in regex if not escaped).
    - \\b ... \\b : "word boundary", ensures that the whole word is detected, not
      just a substring inside another word.
    - re.IGNORECASE : case-insensitive (DÉVERS, Dévers, dévers... are all detected).
    """
    regex_par_categorie = {}
    for categorie, termes in dico.items():
        termes_tries = sorted(termes, key=len, reverse=True)
        motif = r'\b(?:' + '|'.join(re.escape(t) for t in termes_tries) + r')\b'
        regex_par_categorie[categorie] = re.compile(motif, re.IGNORECASE)
    return regex_par_categorie


# Return a dict {category: list of found occurrences} for a text.

def detecter_termes(texte, regex_dict):
    if not isinstance(texte, str):
        return {}
    resultats = {}
    for categorie, pattern in regex_dict.items():
        matches = pattern.findall(texte)
        if matches:
            resultats[categorie] = matches
    return resultats


regex_dict = compiler_dictionnaire(dico_terme_description)

In [5]:
df_annotation = df.copy()

# Count the NUMBER of occurrences rather than using a simple Boolean:
for categorie in dico_terme_description:
    df_annotation[f'nb_{categorie}'] = df_annotation['description'].apply(
        lambda texte: len(detecter_termes(texte, regex_dict).get(categorie, []))
    )

# Build the LIST of occurrences found for each category:
for categorie in dico_terme_description:
    df_annotation[categorie] = df_annotation['description'].apply(
        lambda texte: detecter_termes(texte, regex_dict).get(categorie, [])
    )

## 4. Extracting natural feature sequences

In [6]:
# Create a variable containing the list of categories detected in order of appearance in the text.
def detecter_termes_ordonnes(texte, dico):
    if not isinstance(texte, str):
        return []

    occurrences = []

    for categorie, termes in dico.items():
        for terme in termes:
            motif = re.compile(
                r'\b' + re.escape(terme) + r'\b',
                re.IGNORECASE
            )

            for match in motif.finditer(texte):
                occurrences.append((match.start(), categorie))

    occurrences.sort(key=lambda x: x[0])

    return [categorie for _, categorie in occurrences]


# Add the column without changing existing columns
df_annotation['termes_ordonnes'] = (
    df_annotation['description'].apply(
        lambda texte: detecter_termes_ordonnes(
            texte, dico_terme_description
        )
    )
)

In [7]:
# Create feature transition variables (bigrams)

seq_col = "termes_ordonnes"  
def vers_liste(s):
    if isinstance(s, list):
        return s
    if pd.isna(s):
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [s]
    except (ValueError, SyntaxError):
        return [s]

seqs = df_annotation[seq_col].apply(vers_liste)

# --- 1. Pitch distribution ---
longueurs = seqs.apply(len)
print(longueurs.describe())
print("\nPitch distribution:")
print(longueurs.value_counts().sort_index().head(15))
n2 = (longueurs >= 2).sum()
print(f"\nSequences with >= 2 features (i.e. at least one transition): "
      f"{n2} ({100 * n2 / len(seqs):.1f} %)")

# --- 2. Feature vocabulary ---
vocab = Counter(e for s in seqs for e in s)
print(f"\n{len(vocab)} distinct features; top 30:")
print(vocab.most_common(30))

# --- 3. Transitions (bigrams) ---
bigrams = Counter()
for s in seqs:
    for a, b in zip(s, s[1:]):
        bigrams[(a, b)] += 1
print(f"\n{len(bigrams)} distinct transitions; top 30:")
for (a, b), n in bigrams.most_common(30):
    print(f"  {a} -> {b} : {n}")

count    42861.000000
mean         1.599473
std          1.608853
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         24.000000
Name: termes_ordonnes, dtype: float64

Pitch distribution:
termes_ordonnes
0     11302
1     13456
2      8694
3      4771
4      2326
5      1122
6       592
7       267
8       154
9        83
10       49
11       16
12       10
13        9
14        2
Name: count, dtype: int64

Sequences with >= 2 features (i.e. at least one transition): 18103 (42.2 %)

30 distinct features; top 30:
[('dalle', 11767), ('fissure', 9579), ('traversée', 9541), ('dièdre', 8161), ('surplomb', 6074), ('vire', 4977), ('pilier', 2321), ('cheminée', 2203), ('terrasse', 2099), ('dévers', 1835), ('arête', 1737), ('écailles', 1598), ('bac', 1027), ('éperon', 926), ('couloir', 911), ('trou', 793), ('cannelure', 654), ('réglette', 401), ('goutte_eau', 367), ('fissure-dièdre', 359), ('crochet', 320), ('petite_prise', 288), ('plat', 278), ('i

## 5. Analyzing empty rows to see whether any terms were missed

### 5.1 Frequent words in descriptions where no terms were extracted

In [8]:
# Rows where all categories are empty but the description is not

categories = list(dico_terme_description.keys())

def est_vide(valeur):
    if isinstance(valeur, list):
        return len(valeur) == 0
    return pd.isna(valeur)

toutes_vides = df_annotation[categories].applymap(est_vide).all(axis=1)
description_non_vide = df_annotation['description'].notna() & (df_annotation['description'].str.strip() != "")

lignes_a_investiguer = df_annotation[toutes_vides & description_non_vide]
print(f"Number of rows to investigate: {len(lignes_a_investiguer)}")

C:\Users\lunag\AppData\Local\Temp\ipykernel_26156\1874282961.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  toutes_vides = df_annotation[categories].applymap(est_vide).all(axis=1)


Number of rows to investigate: 11302


In [9]:
# Unigrams

mots_vides = set(stopwords.words('french'))
mots_deja_connus = set()
for termes in dico_terme_description.values():
    for t in termes:
        mots_deja_connus.update(t.lower().split())

def tokeniser(texte):
    return re.findall(r"[a-zàâäéèêëïîôöùûüç]+", texte.lower())

compteur = Counter()
for texte in lignes_a_investiguer['description'].dropna():
    for mot in tokeniser(texte):
        if mot not in mots_vides and mot not in mots_deja_connus and len(mot) > 2:
            compteur[mot] += 1

for mot, freq in compteur.most_common(50):
    print(f"{mot:20s} {freq}")

relais               2075
longueur             2015
puis                 1951
gauche               1933
droite               1905
droit                1362
mur                  1196
plus                 1091
très                 958
départ               942
peu                  911
rocher               880
facile               859
bien                 853
raide                825
tout                 786
dessus               757
jusqu                684
ascendance           664
vers                 586
remonter             549
fin                  533
points               508
belle                502
relai                496
sommet               494
spits                461
voie                 441
rejoindre            435
sup                  398
bloc                 386
courte               359
deux                 353
quelques             351
sortie               343
assez                342
beau                 338
technique            317
point                315
spit             

In [12]:
# Bigrams

compteur_bigrammes = Counter()
for texte in lignes_a_investiguer['description'].dropna():
    mots = [m for m in tokeniser(texte) if m not in mots_vides]
    for bg in bigrams(mots):
        compteur_bigrammes[' '.join(bg)] += 1

for bg, freq in compteur_bigrammes.most_common(50):
    print(f"{bg:30s} {freq}")

tout droit                     661
droit dessus                   394
mur raide                      317
belle longueur                 309
ascendance gauche              277
ascendance droite              274
jusqu relais                   245
vers gauche                    241
plus facile                    241
relais r                       214
vers droite                    203
droite puis                    178
gauche puis                    171
courte longueur                156
dessus relais                  142
départ puis                    132
très belle                     129
puis tout                      126
jusqu sommet                   116
droit puis                     114
petit mur                      104
beau mur                       104
remonter mur                   103
rejoindre relais               102
quelques mètres                101
puis plus                      99
sup e                          97
e sup                          97
peu plus               

In [13]:
# Trigrams

compteur_trigrammes = Counter()
for texte in lignes_a_investiguer['description'].dropna():
    mots = [m for m in tokeniser(texte) if m not in mots_vides]
    for bg in trigrams(mots):
        compteur_trigrammes[' '.join(bg)] += 1

for bg, freq in compteur_trigrammes.most_common(50):
    print(f"{bg:30s} {freq}")

puis tout droit                123
jusqu relais r                 99
sup e sup                      97
très belle longueur            86
sup er sup                     76
puis plus facile               75
droit dessus relais            69
tout droit puis                65
relais r points                54
ascendance vers gauche         53
rejoindre relais r             53
points non reliés              53
gauche puis tout               46
er sup point                   38
r points non                   38
ascendance vers droite         37
droit dessus puis              36
droite puis tout               35
légère ascendance gauche       35
droite puis droit              35
fino alla sosta                35
e sup point                    34
relais rappel r                34
remonter mur raide             33
ascendance gauche puis         30
tout droit jusqu               29
relais chaîné r                29
ascendance droite jusqu        28
puis droit dessus              27
puis ascendan

### 5.2 Tokenization

In [14]:
# --- Step 1: build merging rules for compound terms ---
def construire_motifs_fusion(dico):
    remplacements = []
    for termes in dico.values():
        for t in termes:
            if ' ' in t or "-" in t or "'" in t:
                token_fusionne = re.sub(r"[ '\-]+", "_", t.lower())
                remplacements.append((t, token_fusionne))
    remplacements.sort(key=lambda x: len(x[0]), reverse=True)
    return remplacements

remplacements = construire_motifs_fusion(dico_terme_description)

def preparer_texte(texte):
    """Merge compound terms, then tokenize the text."""
    if not isinstance(texte, str):
        return []
    texte = texte.lower()
    for original, fusionne in remplacements:
        texte = re.sub(re.escape(original.lower()), fusionne, texte)
    return re.findall(r"[a-zàâäéèêëïîôöùûüç_]+", texte)

# --- Step 2: build the training corpus ---
# Use ALL non-empty descriptions (not only the "empty ones to investigate"),
# to give the model as much context as possible
sentences = [
    preparer_texte(texte)
    for texte in df_annotation['description'].dropna()
    if isinstance(texte, str) and texte.strip() != ""
]
sentences = [s for s in sentences if len(s) > 0]
print(f"Number of sentences in the training corpus: {len(sentences)}")

# --- Step 3: train the model ---
model = FastText(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,     # ignore words appearing fewer than 5 times (noise)
    sg=1,
    epochs=15,
    workers=4,
)

# --- Step 4: query the neighbors of each term in the dictionary ---
mots_deja_connus = set()
for termes in dico_terme_description.values():
    for t in termes:
        token = re.sub(r"[ '\-]+", "_", t.lower()) if (' ' in t or '-' in t or "'" in t) else t.lower()
        mots_deja_connus.add(token)

for categorie, termes in dico_terme_description.items():
    for terme in termes:
        token = re.sub(r"[ '\-]+", "_", terme.lower()) if (' ' in terme or '-' in terme or "'" in terme) else terme.lower()
        if token not in model.wv:
            continue
        voisins = model.wv.most_similar(token, topn=10)
        # Keep only neighbors that are NOT already in the dictionary
        nouveaux_candidats = [(v, s) for v, s in voisins if v not in mots_deja_connus]
        if nouveaux_candidats:
            print(f"\n[{categorie}] neighbors of '{terme}' :")
            for v, s in nouveaux_candidats:
                print(f"    {v:20s} {s:.3f}")

Number of sentences in the training corpus: 42811

[dévers] neighbors of 'dévers' :
    déverse              0.676
    surplombe            0.642
    bombement            0.636

[dévers] neighbors of 'déversante' :
    traversante          0.726
    surplombantes        0.644
    fatigante            0.624
    plaisante            0.619

[dévers] neighbors of 'déversant' :
    surplombants         0.674
    raide                0.666
    déverse              0.654

[dévers] neighbors of 'déversantes' :
    surplombantes        0.856
    surplombants         0.730
    rondes               0.730
    riches               0.719
    arquées              0.703
    diverses             0.688
    verticales           0.682

[dévers] neighbors of 'déversants' :
    surplombants         0.859
    surplombantes        0.734
    verticaux            0.694
    pnts                 0.684
    gants                0.679

[dévers] neighbors of 'déversé' :
    déverse              0.744
    vertical    

## 6. Saving the dataset

In [15]:
df_annotation.to_csv("annotation_compte_mots.csv", index=False, encoding="utf-8")